# How a local LLM answers a question using structured data

**You are here → 00 — How a local AI request works.**

**Main workflow:** understand a local AI request → optimize Qwen → optimize Nemotron.
**Optional extensions:** model-specific NIM deployment; separate CPU portability branch.

| **Step** | **Notebook** | **Purpose** |
|---|---|---|
| **→ 0. Understand the request (current)** | [00 — How a local AI request works](00_local_serving.ipynb) | Follow a question through schema context, model serving, and generated SQL, then evaluate the results to establish a quality baseline. |
| 1. Qwen optimization | [01 — Qwen: caching and quantization](01_qwen_cache_quantization.ipynb) | Compare FP16/AWQ W4A16 and prefix caching, with cold and warm requests reported separately. |
| 2. Nemotron optimization | [02 — Nemotron optimization](02_nemotron_optimization.ipynb) | Test schema context, supported prefix caching, concurrent requests, scheduler capacity and output budgets. |
| 3. Qwen NIM (optional) | [03 — Qwen through NIM](03_qwen_nim.ipynb) | Deploy Qwen in a container, check the plan format, and compare schema context and concurrent requests. |
| 4. Nemotron NIM (optional) | [04 — Nemotron through NIM](04_nemotron_nim.ipynb) | Deploy the hybrid model through NIM and compare schema context and request concurrency. |
| Optional CPU portability | [05 — Qwen with ExecuTorch](05_edge_portability_executorch.ipynb) | Follow Qwen3-4B conversion and CPU export, then inspect a generated SQL answer and its timing. |
| Optional compatibility study | [06 — Nemotron with ExecuTorch](06_nemotron_executorch.ipynb) | Check whether the installed exporter accepts Nemotron and identify what a working port still needs. |

**Use Case**: Persona: A clinical trial analyst. Job: Design a clinical study. Asks questions like “How many enrolled subjects are in each treatment arm?” 

The LLM needs more than the question, it needs context: the database structure,
the meaning of “enrolled,” and instructions for writing SQL. The application then
checks the SQL and runs it to produce the answer.

We'll follow the baseline quality path before changing any performance settings. Along the way,
we'll ask how much database context to send, how to serve the model, and how to
tell whether a quick response is also a useful answer. Our examples use **Qwen3-4B
and Nemotron 3 Nano 4B**n, to compare baseline quality in this notebook and later to compare performance optimization and the impact on results through different servings.

**How to run this notebook:** install the [notebook environment](../SETUP.md),
then choose **Run All**. The cells inspect the synthetic database and display
results from experiments already run. They do not ask a model to generate a new
answer, so you can work through the example without downloading weights or using
a GPU. In chapters 01–04, **Replay** reads those saved measurements, **Plan** shows
the settings for a new experiment, and **Live** runs inference on your hardware.



In [1]:
from pathlib import Path
import os
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/clinical_trial_mvp').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from IPython.display import JSON, Markdown, display
from clinical_trial_mvp.tutorial.foundation import inventory, contexts, mac_history, gpu_history, history_table
print('Read-only foundation; mode:', os.environ.get('LOCAL_AI_LAB_MODE', 'replay'))
print('Historical evidence is labeled; no fresh model calls are made.')


Read-only foundation; mode: replay
Historical evidence is labeled; no fresh model calls are made.


### The questions we'll work through

Each section adds one part of the application. The model names stay familiar;
the question we are testing changes.

| Section | What we want to understand |
|---|---|
| 1. Workload | What data and questions must the application handle? |
| 2. Schema context | Does selecting a smaller part of the schema help the model answer? |
| 3. Local serving | What changes when the same model file is served through llama.cpp or Ollama? |
| 4. Several users | Why do GPU servers such as vLLM and SGLang matter when requests overlap? |
| 5. One complete answer | How do a prompt, generated SQL and result check fit together? |

We'll introduce the model format, runtime and hardware with each experiment.
The complete settings are available in the linked experiment reports if you
want to repeat a particular comparison.

**Your TODO — kept for review:**

> TODO: Calling evidence/how to read too much
> TODO see all these below baseline work - so many models -> we need to tell the context or simplify|


## 1. Meet the application and its data

Our application answers questions about subjects, sites, visits and adverse
events in a clinical trial. The database is synthetic; it contains no patient
data. **DuckDB** is the SQL database engine that executes the generated queries.

Here's the path we'll follow:

**Question → schema context → model plan and SQL → validation → DuckDB → result comparison.**

A *schema* describes tables, columns and relationships. A *plan* states what the
answer should count or group before the model writes SQL. Validation checks
whether the generated query is allowed and executable; result comparison checks
its rows against a reference query.

Run the next cell to inspect the ten tables. This notebook reads the supplied
database without changing it. Using the same rows lets us compare the model's
query with the reference consistently.

The evaluation set contains **30 questions**, including **23 with reference SQL**.
The optimization chapters use four questions from that set for shorter experiments.
Repeating those four questions helps measure timing; it doesn't test more kinds
of clinical questions.


In [2]:
database = inventory(ROOT)
display(JSON(database))


<IPython.core.display.JSON object>

## 2. Give the model the context it needs

To write SQL, the model needs table names, columns, joins and the definitions
behind the question. We can prepare that context in two ways:

- **Full schema:** include all the available schema descriptions.
- **Retrieved schema:** select the descriptions that appear relevant to this
  question, using embeddings to find related text.

Both approaches send text to the generation model. Retrieval reduces the amount
of text, but it may also leave out a definition or join the answer needs.

The next cell displays **both prompts for the same question**. Compare the full
prompt with the retrieved prompt: what was kept, and what was removed? The
displayed lengths count characters, not model tokens.

The relevant schema passages have already been selected for these examples.
This cell reads that selection rather than running an embedding model. The
timing comparison that follows measures answer generation and application work;
it excludes the time needed to retrieve those passages.

**Your TODO — kept for review:**

> TODO historical recomputation is too much.


In [3]:
context = contexts(ROOT)
display(JSON({'note': context['note'], 'cases': [
    {key: value for key, value in row.items() if key not in ('full_prompt', 'retrieved_prompt')}
    for row in context['rows']]}))
display(JSON({'full_prompt': context['rows'][0]['full_prompt'],
              'retrieved_prompt': context['rows'][0]['retrieved_prompt']}))


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

### Compare full-schema and retrieved-schema answers

Now let's see what happened when each model answered using those two kinds of
context. This experiment used **Nemotron 3 Nano 4B and Qwen3-4B**, each in
**Q4_K_M GGUF** format through **llama-cpp-python 0.3.35**, on an **Apple M4 Pro
CPU** with GPU offload disabled. GGUF is the model-file format used by llama.cpp;
Q4_K_M identifies the quantized weight representation.

Each model/context configuration answered the 30-question set once. The next
table compares result matches on the 23 questions with reference SQL and
**application latency**: planning, generation, checks and any repair. This is
the time to work through an answer, rather than the time to its first token.

**Your TODO — kept for review:**

> TODO we called last one full prompt. now we are talking about shorter prompt. original MAC result


In [4]:
history = mac_history(ROOT)
display(JSON({'label': history['origin'], 'context_report': history['context']}))


<IPython.core.display.JSON object>

With retrieved schema, Nemotron's median application latency changed from
**79.795 to 29.500 seconds**, while reference-result matches changed from
**10/23 to 8/23**. Qwen's matches changed from **5/23 to 6/23**.

The useful lesson is the trade-off: less input can reduce work, but the removed
text may matter to the answer. Before adopting retrieved context, inspect whether
it preserves the definitions and joins your questions need. These observations
come from this question set and database, so they don't establish a general
accuracy ranking between the models.

See [the full-schema vs. retrieved-schema experiment settings and results](../baseline/mvp1/experiment-tracker/MVP1_Q4_30_CASE_REPORT.md)
for the complete comparison.


## 3. Serve the same model through llama.cpp and Ollama

Once we've chosen a model and prompt, the application needs a way to call the
model. **llama.cpp** gives us direct control over model execution. **Ollama**
provides model management and an API that the application can call.

Here we hold the **Nemotron 3 Nano 4B Q4_K_M GGUF file** fixed and compare those
two serving paths on an **Apple M4 Pro with Metal GPU offload enabled**.
Prompts, seed and output cap match; the measurements report no cached prompt
tokens. There are three measured requests per prompt for each runtime.

Run the next cell to compare **time to first token** and **complete request
time**. This measures a model-generation request. Section 2 measured the broader
application path on CPU, so the two sections answer different timing questions.

You can follow how the application calls each runtime in the
[runtime adapters](../src/clinical_trial_mvp/tutorial/adapters.py) and
[serving harness](../src/clinical_trial_mvp/serving/benchmark.py).

**Your TODO — kept for review:**

> TODO: what is earlier CPU only, this is confusing. remove the line?


In [5]:
display(JSON({'label': history['origin'], 'serving_screen': history['serving'], 'limitations': history['limitations']}))


<IPython.core.display.JSON object>

Across the three measured requests per prompt and runtime, median time to first
token was **0.400 s for llama.cpp** and **0.414 s for Ollama**. Median complete
request time was **1.393 s** and **1.404 s**, respectively. The paired generated
outputs also matched.

For this experiment, timing alone gives little reason to prefer one serving
path. API convenience, model management and runtime controls can guide the
choice. We still need to check the SQL separately: two runtimes producing the
same text doesn't tell us whether that text answers the question.

See [the llama.cpp/Ollama setup, prompts and timing results](../baseline/mvp2/experiment-tracker/mvp2/serving/MILESTONE_REPORT.md).

**Your TODO — kept for review:**

> TODO what is small screen? median time to first token was **0.400 s for llama.cpp**
> TODO: original serving report tag is not intuitive.


## 4. What changes when several users send requests?

The previous section considered individual requests. A service also needs to
handle questions arriving together. **Concurrency** is the number of requests
clients send at once; a serving engine decides how to schedule that work.

This is why we introduce **vLLM and SGLang** here. Both are GPU serving engines
used to explore overlapping requests. The next table shows the existing tests
at **one, two and four concurrent requests** on an **NVIDIA Quadro RTX 5000,
16 GiB VRAM**, using Hugging Face checkpoints with FP16 requested.

Use these tables to understand the progression from one request to a shared
service. This exercise does not declare one engine faster than the other:
the saved summaries lack enough model and request detail for a controlled ranking.
Chapters 01 and 02 continue with vLLM and explicitly vary its optimization settings.

**Your TODO — kept for review:**

> TODO: this is just 00 notebook do we need to say all these? No Dell. we just say which HW everywhere. why SGLang, vllm are shown here?
> TODO: Use them to understand the progression. This exercise does not declare one engine faster over the other.


In [6]:
gpu = gpu_history(ROOT)
display(Markdown(history_table(gpu)))


| Runtime | Clients | TTFT p50 s ≈ | Request E2E p50 s ≈ | Aggregate output tok/s ≈ | Application result match | Evidence |
|---|---|---|---|---|---|---|
| vllm | 1 | 0.19 | 2.63 | 40.67 | 2/4 | historical notebook summary — rounded; pairing unverified |
| vllm | 2 | 0.24 | 3.07 | 67.09 | 2/4 | historical notebook summary — rounded; pairing unverified |
| vllm | 4 | 0.37 | 3.48 | 105.37 | 2/4 | historical notebook summary — rounded; pairing unverified |
| sglang | 1 | 0.13 | 2.47 | 43.49 | 2/4 | historical notebook summary — rounded; pairing unverified |
| sglang | 2 | 0.17 | 2.7 | 75.35 | 2/4 | historical notebook summary — rounded; pairing unverified |
| sglang | 4 | 0.31 | 2.9 | 127.68 | 2/4 | historical notebook summary — rounded; pairing unverified |

Both runtime summaries report **3/4 SQL executions and 2/4 reference-result
matches**. The SQL checks were run serially; the timing tests varied concurrent
requests. Look at those as two related questions: can the application produce
useful SQL, and how does serving behave under load?

Peak GPU memory wasn't captured for these tests. The next optimization chapters
show more detailed request settings and results so we can examine individual
changes more closely.

The [vLLM/SGLang test details and available measurements](../baseline/mvp3/tutorial/RESULTS_PORTABILITY.md)
describe the scope of this comparison.


## 5. Follow one question through to its answer

Let's connect the pieces with a **Qwen3-4B FP16 request through vLLM on an NVIDIA
Quadro RTX 5000, 16 GiB VRAM**. This is the baseline you'll inspect in chapter 01.
The next cell displays the request settings and prompt, then the model's plan,
generated SQL, execution attempts and reference-result comparison.

First identify what the question asks. Then check whether the SQL keeps its
filters and grouping, and counts the intended subjects or events. Finally,
compare the returned rows with the reference. A query can execute successfully
without answering the question correctly.

This cell displays a worked example from the saved baseline; it sends no new
inference request. Running it again displays the same example. A new Live model
run may generate different text, which is why we evaluate its answer again.

The implementation is available in the [request builder](../src/clinical_trial_mvp/serving/client.py),
[plan and SQL checks](../src/clinical_trial_mvp/agent/analyst.py), and
[result comparison code](../src/clinical_trial_mvp/evaluation/scorers.py).

**Your TODO — kept for review:**

> TODO: Nov 6. dell machine should not be mentioned. reconstruction keep somewhere in gitignore as notes.md file, user does not care about reconsturction, every iteration in llm will not give same result anyway.


In [7]:
from clinical_trial_mvp.tutorial.workflow import Study, template
from clinical_trial_mvp.tutorial.teaching import request_example, response_example
config = template('vllm', 'qwen')
config['mode'] = 'replay'
example = Study('vllm', 'qwen', root=ROOT, config=config)
example.run('fp16_cache_off')
display(JSON(request_example(example, 'fp16_cache_off')))
display(JSON(response_example(example, 'fp16_cache_off')))


<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

## 6. Choose what to measure before optimizing

We can now separate three parts of the application:

| Part | What we'll measure | Why it matters |
|---|---|---|
| Serving | Time to first token, complete request time, output tokens per second | How quickly the service responds and how much work it handles |
| Generation | Input/output token counts and truncated responses | How much text the model processes and produces |
| Application | Valid plans, executable SQL, reference-result matches and repairs | Whether the service produces a usable answer |

The optimization chapters compare these together. A faster first token doesn't
guarantee a faster complete answer, and faster SQL generation doesn't guarantee
correct SQL. Reference-result matches are useful automated checks; reviewing the
query's meaning remains a separate task.

We'll keep the model, prompts, data, precision and request settings visible while
changing one factor at a time. Model startup is measured separately from requests.

Continue to [01 — Qwen caching and quantization](01_qwen_cache_quantization.ipynb),
then [02 — Nemotron optimization](02_nemotron_optimization.ipynb).
Chapters [03](03_qwen_nim.ipynb) and [04](04_nemotron_nim.ipynb) explore NIM deployment.
For setup or execution help, use [installation](../SETUP.md) and
[run instructions](../tutorial/RUN_ALL.md).
